# 📖 Story Generation Pipeline — Google Colab

**Four-stage pipeline:** User Prompt → MoPS Premise → DOME Memory → T5 Outline → BART Story

| Stage | Module | Model |
|-------|--------|-------|
| 1 | Premise expansion (MoPS-inspired) | Template-based |
| 2 | Memory extraction (DOME-inspired) | spaCy NER |
| 3 | Outline generation (EtriCA-inspired) | T5-small |
| 4 | Story generation (Hierarchical) | BART-base |

### Notebook sections
1. ⚙️ Setup — clone repo & install dependencies
2. 📊 Data Preparation (ROCStories + WritingPrompts)
3. 🏋️ Training (ROCStories + WritingPrompts)
4. 📏 Comprehensive Evaluation (ROUGE-L / BLEU / METEOR / BERTScore)
5. 🔬 Ablation Study (6 conditions)
6. 🎨 Interactive Story Generation Demo
7. 💾 Download Results

---
> **Before you start:** `Runtime → Change runtime type → T4 GPU`  
> **Note:** Colab sessions are ephemeral. Trained models and processed data are stored in `/content/` and will be lost when the session ends. Download anything you want to keep (Section 7).

---
## ⚙️ 1. Setup

Run these three cells at the start of every session.

In [14]:
# ── 1a. Check GPU ─────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU : {gpu}")
    print(f"VRAM: {vram:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > GPU (T4)")

GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


In [16]:
# ── 1b. Clone repository from GitHub ─────────────────────────────────────
GITHUB_REPO = "https://github.com/Ardameliksah/StoryGeneration.git"
BRANCH      = "mert_test"   # <- branch where all code lives
REPO_DIR    = "/content/StoryGeneration"

import os, pathlib

if pathlib.Path(REPO_DIR).exists():
    # Already cloned — pull latest on the correct branch
    !git -C {REPO_DIR} fetch --quiet
    !git -C {REPO_DIR} checkout --quiet {BRANCH}
    !git -C {REPO_DIR} pull --quiet
    print(f"Repo updated ({BRANCH}) at {REPO_DIR}")
else:
    !git clone --quiet --branch {BRANCH} {GITHUB_REPO} {REPO_DIR}
    print(f"Repo cloned ({BRANCH}) to {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Verify required source files
REQUIRED = [
    "inference.py", "metrics.py", "evaluate.py", "ablation.py",
    "train_outline.py", "train_story.py",
    "prepare_data.py", "prepare_writingprompts.py",
]
missing = [f for f in REQUIRED if not pathlib.Path(f).exists()]
if missing:
    print("MISSING files:", missing)
else:
    print("All source files present")

Repo updated (mert_test) at /content/StoryGeneration
Working directory: /content/StoryGeneration
All source files present


In [17]:
# ── 1c. Install dependencies (~2–3 min, run once per session) ─────────────
!pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.30.0 \
    sentencepiece>=0.2.0 \
    rouge-score>=0.1.2 \
    sacrebleu>=2.4.0 \
    nltk>=3.8.0 \
    bert-score>=0.3.13 \
    spacy>=3.7.0 \
    datasets>=2.19.0

!python -m spacy download en_core_web_sm -q

import nltk
for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

print("All dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 198.4 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All dependencies installed


---
## 📊 2. Data Preparation

### 2a. ROCStories

ROCStories requires a manual upload because the dataset is not publicly downloadable via an API.  
Upload `rocstories_train.txt` and `rocstories_test.txt` using the cell below.

In [18]:
# ── 2a-i. Upload ROCStories raw files ────────────────────────────────────
import os, pathlib
from google.colab import files

os.makedirs("data/raw", exist_ok=True)

train_ok = pathlib.Path("data/raw/rocstories_train.txt").exists()
test_ok  = pathlib.Path("data/raw/rocstories_test.txt").exists()

if train_ok and test_ok:
    print("ROCStories files already present — skipping upload")
else:
    print("Select both rocstories_train.txt and rocstories_test.txt ...")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = pathlib.Path("data/raw") / fname
        dest.write_bytes(data)
        print(f"  Saved -> {dest}")

ROCStories files already present — skipping upload


In [19]:
# ── 2a-ii. Build ROCStories JSONL files ───────────────────────────────────
# Output: data/processed/outline_{train,val,test}.jsonl
#         data/processed/story_{train,val,test}.jsonl
import pathlib

if pathlib.Path("data/processed/story_train.jsonl").exists():
    print("Processed ROCStories data already exists — skipping")
else:
    !python prepare_data.py

Loading data/raw/rocstories_train.txt ...
  train: 69919 examples
  val:   7769 examples
Loading data/raw/rocstories_test.txt ...
  test:  19410 examples
Done.


### 2b. WritingPrompts *(optional — only needed for WP fine-tuning)*

Downloads `euclaise/writingprompts` from HuggingFace automatically.  
Full dataset ~272K stories (~20 min). Use `MAX_WP_STORIES` to limit for quick experiments.

In [20]:
# ── 2b. Build WritingPrompts JSONL files ──────────────────────────────────
import pathlib

MAX_WP_STORIES = 50000   # None = full ~272K dataset

if pathlib.Path("data/processed/wp_story_train.jsonl").exists():
    print("Processed WritingPrompts data already exists — skipping")
else:
    flag = f"--max-stories {MAX_WP_STORIES}" if MAX_WP_STORIES else ""
    !python prepare_writingprompts.py {flag}

README.md: 100% 837/837 [00:00<00:00, 8.69MB/s]
data/train-00000-of-00002-105e07cb0d1994(…): 100% 272M/272M [00:02<00:00, 113MB/s] 
data/train-00001-of-00002-4fdb982c110564(…): 100% 272M/272M [00:02<00:00, 124MB/s] 
data/test-00000-of-00001-16503b0c26ed00c(…): 100% 30.0M/30.0M [00:00<00:00, 37.2MB/s]
data/validation-00000-of-00001-137b93e1e(…): 100% 30.7M/30.7M [00:00<00:00, 75.7MB/s]
Generating train split: 100% 272600/272600 [00:01<00:00, 144389.88 examples/s]
Generating test split: 100% 15138/15138 [00:00<00:00, 203100.81 examples/s]
Generating validation split: 100% 15620/15620 [00:00<00:00, 201168.75 examples/s]
Total: 50000 | Train: 45000 | Val: 2500 | Test: 2500
Processing train ...
  train: 32213 examples written
Processing val ...
  val: 1800 examples written
Processing test ...
  test: 1783 examples written
Done.


### 2c. Premise-augmented ROCStories *(needed for the premise ablation)*

Reads the existing ROCStories JSONL files and rewrites each training input to
include the structured premise slot. Creates a separate dataset for training
the premise-aware BART model. Requires Section 2a to have run first.

In [ ]:
# Build premise-augmented story JSONL files
# Input:  'title outline: e1 | e2 | e3'
# Output: 'title premise: Title:...\nSetting:... outline: e1 | e2 | e3'
import json, pathlib, sys
sys.path.insert(0, '/content/StoryGeneration')
from inference import expand_premise

def add_premise(src, dst):
    written = 0
    with open(src, encoding='utf-8') as fin, open(dst, 'w', encoding='utf-8') as fout:
        for line in fin:
            ex      = json.loads(line)
            title   = ex['input'].split(' outline:')[0]
            outline = ex['input'].split(' outline:')[1]
            premise = expand_premise(title)
            new_in  = f'{title} premise: {premise} outline:{outline}'
            fout.write(json.dumps({'input': new_in, 'target': ex['target']}) + '\n')
            written += 1
    print(f'  {src} -> {dst}  ({written} examples)')

pathlib.Path('data/processed').mkdir(parents=True, exist_ok=True)

if pathlib.Path('data/processed/story_premise_train.jsonl').exists():
    print('Premise-augmented data already exists - skipping')
else:
    add_premise('data/processed/story_train.jsonl', 'data/processed/story_premise_train.jsonl')
    add_premise('data/processed/story_val.jsonl',   'data/processed/story_premise_val.jsonl')
    print('Done.')


---
## 🏋️ 3. Training

### 3a. T5-small — Outline Generator (ROCStories)

Learns: `story title → event1 | event2 | event3`  
~15 min / epoch on T4.

In [21]:
# ── Train from scratch (3 epochs recommended) ─────────────────────────────
!python train_outline.py --data roc --epochs 3

Device: cuda | Dataset: roc | Grad accum: 1
tokenizer_config.json: 2.32kB [00:00, 14.1MB/s]
spiece.model: 100% 792k/792k [00:00<00:00, 2.66MB/s]
tokenizer.json: 1.39MB [00:00, 5.93MB/s]
config.json: 1.21kB [00:00, 9.49MB/s]
model.safetensors: 100% 242M/242M [00:01<00:00, 200MB/s] 
Loading weights: 100% 131/131 [00:00<00:00, 1363.33it/s, Materializing param=shared.weight]                                                     
generation_config.json: 100% 147/147 [00:00<00:00, 1.95MB/s]
  Epoch 1 | step 200/4370 | loss 4.5826
  Epoch 1 | step 400/4370 | loss 4.2555
  Epoch 1 | step 600/4370 | loss 3.9195
  Epoch 1 | step 800/4370 | loss 3.6406
  Epoch 1 | step 1000/4370 | loss 3.4337
  Epoch 1 | step 1200/4370 | loss 3.2825
  Epoch 1 | step 1400/4370 | loss 3.1613
  Epoch 1 | step 1600/4370 | loss 3.0650
  Epoch 1 | step 1800/4370 | loss 2.9822
  Epoch 1 | step 2000/4370 | loss 2.9163
  Epoch 1 | step 2200/4370 | loss 2.8606
  Epoch 1 | step 2400/4370 | loss 2.8113
  Epoch 1 | step 2600/43

### 3b. BART-base — Story Generator (ROCStories)

Learns: `title outline: [events] [MEM] ... → full story`  
~25 min / epoch on T4.

In [22]:
# ── Train from scratch (3 epochs recommended) ─────────────────────────────
!python train_story.py --data roc --epochs 3

Device: cuda | Dataset: roc | Grad accum: 1
vocab.json: 899kB [00:00, 56.6MB/s]
merges.txt: 456kB [00:00, 184MB/s]
tokenizer.json: 1.36MB [00:00, 208MB/s]
config.json: 1.72kB [00:00, 15.3MB/s]
model.safetensors: 100% 558M/558M [00:03<00:00, 164MB/s]  
Loading weights: 100% 259/259 [00:00<00:00, 936.13it/s, Materializing param=model.shared.weight]                                   s.4.self_attn.out_proj.weight]
  Epoch 1 | step 200/8740 | loss 2.5418
  Epoch 1 | step 400/8740 | loss 2.2611
  Epoch 1 | step 600/8740 | loss 2.1302
  Epoch 1 | step 800/8740 | loss 2.0505
  Epoch 1 | step 1000/8740 | loss 1.9925
  Epoch 1 | step 1200/8740 | loss 1.9524
  Epoch 1 | step 1400/8740 | loss 1.9176
  Epoch 1 | step 1600/8740 | loss 1.8906
  Epoch 1 | step 1800/8740 | loss 1.8688
  Epoch 1 | step 2000/8740 | loss 1.8499
  Epoch 1 | step 2200/8740 | loss 1.8317
  Epoch 1 | step 2400/8740 | loss 1.8171
  Epoch 1 | step 2600/8740 | loss 1.8055
  Epoch 1 | step 2800/8740 | loss 1.7925
  Epoch 1 | step

### 3d. BART-base — Premise-trained Story Generator (ROCStories)

Trains a **separate** BART on premise-augmented inputs so the model learns
to actually use the premise text. Saves to `models/bart_story_premise/`.
Does NOT overwrite the standard `models/bart_story/` checkpoint.
~25 min / epoch on T4. Run Section 2c first.

In [ ]:
# Train BART with premise in the input (ROCStories)
# Saves to models/bart_story_premise/ -- does NOT overwrite models/bart_story/
!python train_story.py --data roc_premise --epochs 3


### 3c. WritingPrompts Fine-tuning *(optional)*

`--grad-accum 4` simulates batch size 32 on T4 (WP stories are longer, so the real batch must be smaller).

In [23]:
# ── Train T5 outline on WritingPrompts ────────────────────────────────────
!python train_outline.py --data wp --epochs 3 --grad-accum 4

# ── Train BART story on WritingPrompts ────────────────────────────────────
!python train_story.py --data wp --epochs 3 --grad-accum 4

Device: cuda | Dataset: wp | Grad accum: 4
Loading weights: 100% 131/131 [00:00<00:00, 1923.01it/s, Materializing param=shared.weight]                                                      
  Epoch 1 | step 200/4027 | loss 6.3028
  Epoch 1 | step 400/4027 | loss 6.2241
  Epoch 1 | step 600/4027 | loss 6.1203
  Epoch 1 | step 800/4027 | loss 5.9876
  Epoch 1 | step 1000/4027 | loss 5.8626
  Epoch 1 | step 1200/4027 | loss 5.7379
  Epoch 1 | step 1400/4027 | loss 5.6204
  Epoch 1 | step 1600/4027 | loss 5.5133
  Epoch 1 | step 1800/4027 | loss 5.4223
  Epoch 1 | step 2000/4027 | loss 5.3444
  Epoch 1 | step 2200/4027 | loss 5.2750
  Epoch 1 | step 2400/4027 | loss 5.2166
  Epoch 1 | step 2600/4027 | loss 5.1660
  Epoch 1 | step 2800/4027 | loss 5.1203
  Epoch 1 | step 3000/4027 | loss 5.0781
  Epoch 1 | step 3200/4027 | loss 5.0439
  Epoch 1 | step 3400/4027 | loss 5.0112
  Epoch 1 | step 3600/4027 | loss 4.9825
  Epoch 1 | step 3800/4027 | loss 4.9542
  Epoch 1 | step 4000/4027 | loss 4.

---
## 📏 4. Comprehensive Evaluation

| Metric | What it captures |
|--------|------------------|
| **ROUGE-L** | Longest common subsequence overlap |
| **BLEU** | n-gram precision (sacrebleu) |
| **METEOR** | Precision + recall with stemming & synonym matching |
| **BERTScore** | Semantic similarity via contextual embeddings |

> Low BLEU/ROUGE is expected for open-ended story generation.

In [24]:
# ── ROCStories — validation split ────────────────────────────────────────
!python evaluate.py --data roc --n 500 --split val


Evaluating 500 examples from data/processed/story_val.jsonl
  Memory module : ON
  BERTScore model: distilbert-base-uncased
[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...
Loading weights: 100% 131/131 [00:00<00:00, 1456.21it/s, Materializing param=shared.weight]                                                     
[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...
Loading weights: 100% 260/260 [00:00<00:00, 1144.34it/s, Materializing param=model.shared.weight]                                  
  generated 50/500 stories …
  generated 100/500 stories …
  generated 150/500 stories …
  generated 200/500 stories …
  generated 250/500 stories …
  generated 300/500 stories …
  generated 350/500 stories …
  generated 400/500 stories …
  generated 450/500 stories …
  generated 500/500 stories …

Computing metrics:
  ROUGE-L   … 0.3175
  BLEU      … 18.06
  METEOR    … 0.3131
  BERTScore … Warning: You are sending unauthent

In [25]:
# ── ROCStories — test split (final / paper numbers) ───────────────────────
!python evaluate.py --data roc --n 500 --split test


Evaluating 500 examples from data/processed/story_test.jsonl
  Memory module : ON
  BERTScore model: distilbert-base-uncased
[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...
Loading weights: 100% 131/131 [00:00<00:00, 1442.36it/s, Materializing param=shared.weight]                                                     
[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...
Loading weights: 100% 260/260 [00:00<00:00, 1001.48it/s, Materializing param=model.shared.weight]                                  
  generated 50/500 stories …
  generated 100/500 stories …
  generated 150/500 stories …
  generated 200/500 stories …
  generated 250/500 stories …
  generated 300/500 stories …
  generated 350/500 stories …
  generated 400/500 stories …
  generated 450/500 stories …
  generated 500/500 stories …

Computing metrics:
  ROUGE-L   … 0.3140
  BLEU      … 17.68
  METEOR    … 0.3088
  BERTScore … Warning: You are sending unauthen

In [ ]:
# ── WritingPrompts — validation split (requires Section 3c) ──────────────
!python evaluate.py --data wp --n 200 --split val

In [26]:
# ── Python API — returns a dict, useful for building tables ───────────────
import importlib, sys
sys.path.insert(0, "/content/StoryGeneration")

import evaluate as eval_mod
importlib.reload(eval_mod)

roc_val = eval_mod.evaluate(
    data="roc", split="val", n=100,
    use_memory=True,
    bert_model="distilbert-base-uncased",
)
print(roc_val)


Evaluating 100 examples from data/processed/story_val.jsonl
  Memory module : ON
  BERTScore model: distilbert-base-uncased
[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

  generated 50/100 stories …
  generated 100/100 stories …

Computing metrics:
  ROUGE-L   … 0.3183
  BLEU      … 18.33
  METEOR    … 0.3161
  BERTScore … 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.8136

  Dataset   : ROC | split=val | n=100
  Memory    : on
  ROUGE-L   : 0.3183
  BLEU      : 18.33
  METEOR    : 0.3161
  BERTScore : 0.8136
Note: low BLEU/ROUGE is normal for open-ended story generation.
{'rouge_l': 0.31827067774765366, 'bleu': 18.33343580546529, 'meteor': 0.3161404569931683, 'bertscore': 0.8136333227157593}


---
## 🔬 5. Ablation Study

Six conditions isolate the contribution of each pipeline component:

| Condition | What changes | Tests |
|-----------|-------------|-------|
| `full` | — baseline — | — |
| `no_memory` | DOME `[MEM]` block removed | memory module |
| `no_outline` | no outline fed to BART | outline module |
| `with_premise` | premise text added to BART input | premise contribution |
| `sent_outline` | oracle sentences from reference as outline | event vs. sentence format + ceiling |
| `wp_models` | WP-trained models evaluated on WP val | dataset effect |

In [27]:
# ── Quick smoke test (50 examples, ~10 min on T4) ─────────────────────────
!python ablation.py --n 50


── Condition: full ────────────────────────────────────
[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...
Loading weights: 100% 131/131 [00:00<00:00, 1247.53it/s, Materializing param=shared.weight]                                                      
[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...
Loading weights: 100% 260/260 [00:00<00:00, 943.49it/s, Materializing param=model.shared.weight]                                   
    [full] 50/50 done …
  Computing metrics …
  ROUGE-L   … 0.3120
  BLEU      … 18.45
  METEOR    … 0.3118
  BERTScore … Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100% 100/100 [00:00<00:00, 1387.73it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
-------------

In [28]:
# ── Full ablation (200 examples per condition, ~40 min on T4) ─────────────
!python ablation.py --n 200 --split val


── Condition: full ────────────────────────────────────
[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...
Loading weights: 100% 131/131 [00:00<00:00, 1216.28it/s, Materializing param=shared.weight]                                                      
[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...
Loading weights: 100% 260/260 [00:00<00:00, 933.64it/s, Materializing param=model.shared.weight]                                   
    [full] 50/200 done …
    [full] 100/200 done …
    [full] 150/200 done …
    [full] 200/200 done …
  Computing metrics …
  ROUGE-L   … 0.3208
  BLEU      … 18.40
  METEOR    … 0.3153
  BERTScore … Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100% 100/100 [00:00<00:00, 1910.83it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: d

In [29]:
# ── Python API — run ablation and render a highlighted DataFrame ───────────
import importlib, sys
sys.path.insert(0, "/content/StoryGeneration")

import ablation as ab
importlib.reload(ab)

import pandas as pd

abl_results = ab.ablation(
    n=100,
    split="val",
    bert_model="distilbert-base-uncased",
)

df = pd.DataFrame(abl_results).T
df.index.name = "condition"
df.columns    = ["ROUGE-L", "BLEU", "METEOR", "BERTScore"]
df = df.round(4)
display(df.style.highlight_max(axis=0, color="#c6efce"))


── Condition: full ────────────────────────────────────
[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

    [full] 50/100 done …
    [full] 100/100 done …
  Computing metrics …
  ROUGE-L   … 0.3183
  BLEU      … 18.33
  METEOR    … 0.3161
  BERTScore … 

...

    [wp_models] 50/100 done …
    [wp_models] 100/100 done …
  Computing metrics …
  ROUGE-L   … 0.1410
  BLEU      … 2.56
  METEOR    … 0.2031
  BERTScore … 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.7180


                            ABLATION RESULTS                            
                           n=100 | split=val                            
Condition           ROUGE-L     BLEU    METEOR   BERTScore
------------------------------------------------------------------------
full                 0.3183     18.33     0.3161       0.8136 
no_memory            0.3134     17.47     0.3374       0.8142 
no_outline           0.3151     17.88     0.3354       0.8162 
with_premise         0.2492     11.63     0.2695       0.7641 
sent_outline        *0.5879*   *44.43*   *0.6197*     *0.9016*
wp_models            0.1410      2.56     0.2031       0.7180 
  * = best in column

Condition legend:
  full         – T5 event outline + DOME memory (ROC models)
  no_memory    – DOME memory module removed
  no_outline   – no outline fed to story model
  with_premise – structured premise prepended to BART input
  sent_outline – oracle sentence outline (upper bound)
  wp_models    – WritingProm

,ROUGE-L,BLEU,METEOR,BERTScore
condition,,,,
full,0.318300,18.333400,0.316100,0.813600
no_memory,0.313400,17.472900,0.337400,0.814200
no_outline,0.315100,17.880200,0.335400,0.816200
with_premise,0.249200,11.632800,0.269500,0.764100
sent_outline,0.587900,44.428300,0.619700,0.901600
wp_models,0.141000,2.555800,0.203100,0.718000


---
## 5b. WritingPrompts Ablation

Same four conditions as the ROC ablation, using WP-trained models on WP val data.
Requires Section 3c (WP training) to have run.

| Condition | What changes |
|-----------|-------------|
| `full` | WP baseline: T5 event outline + DOME memory |
| `no_memory` | Remove DOME memory |
| `no_outline` | Remove outline |
| `sent_outline` | Oracle sentence outline (upper bound) |

In [ ]:
# Full WritingPrompts ablation (200 examples, ~40 min on T4)
!python ablation.py --data wp --n 200 --split val


In [ ]:
# Python API -- WP ablation as a highlighted DataFrame
import importlib, sys
sys.path.insert(0, '/content/StoryGeneration')
import ablation as ab
importlib.reload(ab)
import pandas as pd

wp_results = ab.ablation(n=100, split='val', data='wp')

df = pd.DataFrame(wp_results).T
df.index.name = 'condition'
df.columns    = ['ROUGE-L', 'BLEU', 'METEOR', 'BERTScore']
display(df.round(4).style.highlight_max(axis=0, color='#c6efce'))


---
## 5c. Premise Ablation (Fair Comparison)

Three conditions that isolate the true effect of premise:

| Condition | BART model | Premise in input? | Fair? |
|-----------|-----------|------------------|-------|
| `full` | `bart_story` | No | Yes |
| `with_premise` | `bart_story` | Yes | No -- model never saw premise |
| `premise_trained` | `bart_story_premise` | Yes | Yes -- model trained with premise |

Requires Section 3d (premise model training) to have run.

In [ ]:
# Premise ablation -- compare full vs with_premise vs premise_trained
!python ablation.py --data roc --n 200 --conditions full with_premise premise_trained


In [ ]:
# Python API -- premise ablation as a highlighted DataFrame
import importlib, sys
sys.path.insert(0, '/content/StoryGeneration')
import ablation as ab
importlib.reload(ab)
import pandas as pd

premise_results = ab.ablation(
    n=100, split='val', data='roc',
    conditions=['full', 'with_premise', 'premise_trained'],
)

df = pd.DataFrame(premise_results).T
df.index.name = 'condition'
df.columns    = ['ROUGE-L', 'BLEU', 'METEOR', 'BERTScore']
display(df.round(4).style.highlight_max(axis=0, color='#c6efce'))


---
## 🎨 6. Interactive Story Generation Demo

In [30]:
# ── Generate a story from a custom prompt ────────────────────────────────
import sys
sys.path.insert(0, "/content/StoryGeneration")

import inference

# 'roc' = ROCStories models  |  'wp' = WritingPrompts models
MODEL = "roc"

inference.T5_CHECKPOINT   = f"models/t5_outline{'_wp' if MODEL == 'wp' else ''}"
inference.BART_CHECKPOINT = f"models/bart_story{'_wp' if MODEL == 'wp' else ''}"
inference._outline_cache  = None
inference._story_cache    = None

PROMPT = "She finally found what she had been looking for"   # <- change me

result = inference.run_pipeline(PROMPT)


Prompt: She finally found what she had been looking for

[Memory Notes]
(none extracted)

[Stage 1 — Premise]
Title: She finally found what she had been looking for
Setting: An everyday environment relevant to the title.
Main character: A person affected by the situation in the title.
Goal: To resolve or respond to the situation described.
Conflict: An unexpected complication arises along the way.
[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


[Stage 2 — Outline]
She found what she had been looking for | She decided to try to find a new one | she found the perfect one for her new job's
[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


[Stage 3 — Story]
She finally found what she had been looking for. It was a brand new car. She decided to try to find a new one. After searching for a few hours, she finally gave up. Luckily she found the perfect one for her new job's car payment plan.Sara was so happy with the car she was able to pay for it.



In [31]:
# ── Side-by-side comparison of pipeline variants ─────────────────────────
import inference
from inference import generate_outline, generate_story, extract_memory, expand_premise

PROMPT = "The last train left without him"   # <- change me

outline = generate_outline(PROMPT)
memory  = extract_memory(PROMPT)
premise = expand_premise(PROMPT)

print(f"Prompt  : {PROMPT}")
print(f"Outline : {outline}")
print(f"Memory  : {memory or '(none extracted)'}")
print()

variants = {
    "Full pipeline" : dict(use_memory=True,  use_outline=True,  use_premise=False),
    "No memory"     : dict(use_memory=False, use_outline=True,  use_premise=False),
    "No outline"    : dict(use_memory=False, use_outline=False, use_premise=False),
    "With premise"  : dict(use_memory=True,  use_outline=True,  use_premise=True),
}

for label, flags in variants.items():
    story = generate_story(PROMPT, outline, memory, premise=premise, **flags)
    print(f"── {label} ──")
    print(story)
    print()

Prompt  : The last train left without him
Outline : The last train left without him | he walked to the hospital | The train's owner slammed the train for a while
Memory  : [MEM] objects: train

── Full pipeline ──
The last train left without him. He was very sick. So he walked to the hospital. The doctor told him he was in a bad condition and he needed to be hospitalized for a while.The train's owner slammed the train door shut. Now MEM objects to being on a train with a broken door.

── No memory ──
The last train left without him. He was very sick. So he walked to the hospital to get checked out. The doctor told him he was in a bad condition and he needed to be hospitalized! The train's owner slammed the train for a while and gave him a ticket for the rest of the trip.

── No outline ──
The last train left without him. It was the last time he would be on the train. The last one left with him, and he was sad to be gone. But then the next train came back, carrying him with them again, 

In [32]:
# ── Batch generation ──────────────────────────────────────────────────────
import inference
from inference import generate_outline, generate_story, extract_memory

inference._outline_cache = None
inference._story_cache   = None

prompts = [
    "She finally found what she had been looking for",
    "The dog had been waiting at the door for three days",
    "He opened the letter and his hands began to shake",
    "The old library held a secret no one had discovered",
    "They said it was impossible, but she proved them wrong",
]

stories = []
for p in prompts:
    outline = generate_outline(p)
    memory  = extract_memory(p)
    story   = generate_story(p, outline, memory)
    stories.append({"prompt": p, "outline": outline, "story": story})
    print(f"done: {p[:55]}")

# Show sample
s = stories[0]
print(f"\nPrompt  : {s['prompt']}")
print(f"Outline : {s['outline']}")
print(f"Story   : {s['story']}")

[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

done: She finally found what she had been looking for
done: The dog had been waiting at the door for three days
done: He opened the letter and his hands began to shake
done: The old library held a secret no one had discovered
done: They said it was impossible, but she proved them wrong

Prompt  : She finally found what she had been looking for
Outline : She found what she had been looking for | She decided to try to find a new one | she found the perfect one for her new job's
Story   : She finally found what she had been looking for. It was a brand new car. She decided to try to find a new one. After searching for a few hours, she finally gave up. Luckily she found the perfect one for her new job's car payment plan.Sara was so happy with the car she was able to pay for it.


In [33]:
# ── Score a hypothesis against a reference ────────────────────────────────
import metrics

hypothesis = "She searched for years until she stumbled upon a small shop. Inside was the necklace her grandmother had lost. The shopkeeper smiled as if he had been expecting her. She paid with the last of her savings and walked home in tears."
reference  = "She had been looking for the missing heirloom her whole life. One afternoon she found it in an antique market downtown. The seller told her it had been there for twenty years."

scores = metrics.compute_all([hypothesis], [reference], verbose=True)
print(scores)

  ROUGE-L   … 0.1892
  BLEU      … 3.13
  METEOR    … 0.2746
  BERTScore … 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.7886
{'rouge_l': 0.18918918918918917, 'bleu': 3.128131110780681, 'meteor': 0.27461204458971145, 'bertscore': 0.7885687351226807}


---
## 💾 7. Download Results

Runs a full evaluation + ablation and downloads JSON/CSV files to your computer.

In [34]:
import importlib, json, pathlib, sys, datetime
import pandas as pd
from google.colab import files

sys.path.insert(0, '/content/StoryGeneration')

# Timestamp so repeated runs never overwrite previous results
ts  = datetime.datetime.now().strftime('%Y%m%d_%H%M')
out = pathlib.Path('results')
out.mkdir(exist_ok=True)
to_download = []

# ROC evaluation
import evaluate as eval_mod
importlib.reload(eval_mod)
roc_scores = eval_mod.evaluate(data='roc', split='val', n=500)
f = out / f'evaluation_roc_{ts}.json'
f.write_text(json.dumps(roc_scores, indent=2))
pd.DataFrame([roc_scores], index=['ROCStories']).to_csv(out / f'evaluation_roc_{ts}.csv')
to_download += [f, out / f'evaluation_roc_{ts}.csv']

# ROC ablation (all 7 conditions)
import ablation as ab
importlib.reload(ab)
roc_abl = ab.ablation(n=200, split='val', data='roc')
f = out / f'ablation_roc_{ts}.json'
f.write_text(json.dumps(roc_abl, indent=2))
pd.DataFrame(roc_abl).T.to_csv(out / f'ablation_roc_{ts}.csv')
to_download += [f, out / f'ablation_roc_{ts}.csv']

# WP ablation (if WP models trained)
if pathlib.Path('models/bart_story_wp/config.json').exists():
    wp_abl = ab.ablation(n=200, split='val', data='wp')
    f = out / f'ablation_wp_{ts}.json'
    f.write_text(json.dumps(wp_abl, indent=2))
    pd.DataFrame(wp_abl).T.to_csv(out / f'ablation_wp_{ts}.csv')
    to_download += [f, out / f'ablation_wp_{ts}.csv']
else:
    print('WP models not found - skipping WP ablation')

# Premise ablation (if premise model trained)
if pathlib.Path('models/bart_story_premise/config.json').exists():
    pr_abl = ab.ablation(n=200, split='val', data='roc',
                         conditions=['full', 'with_premise', 'premise_trained'])
    f = out / f'ablation_roc_premise_{ts}.json'
    f.write_text(json.dumps(pr_abl, indent=2))
    pd.DataFrame(pr_abl).T.to_csv(out / f'ablation_roc_premise_{ts}.csv')
    to_download += [f, out / f'ablation_roc_premise_{ts}.csv']
else:
    print('Premise model not found - skipping premise ablation')

# Display
print('\n=== ROC Evaluation ===')
display(pd.DataFrame([roc_scores], index=['ROCStories']).round(4))
print('\n=== ROC Ablation ===')
display(pd.DataFrame(roc_abl).T.round(4).style.highlight_max(axis=0, color='#c6efce'))

# Download all result files
print(f'\nDownloading {len(to_download)} files ...')
for f in to_download:
    files.download(str(f))
    print(f'  {f.name}')



Evaluating 500 examples from data/processed/story_val.jsonl
  Memory module : ON
  BERTScore model: distilbert-base-uncased
[Outline model] Loading T5-small from models/t5_outline (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

[Story model]   Loading BART-base from models/bart_story (fine-tuned checkpoint) ...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

  generated 50/500 stories …
  generated 100/500 stories …
  generated 150/500 stories …
  generated 200/500 stories …
  generated 250/500 stories …
  generated 300/500 stories …
  generated 350/500 stories …
  generated 400/500 stories …
  generated 450/500 stories …
  generated 500/500 stories …

Computing metrics:
  ROUGE-L   … 0.3175
  BLEU      … 18.06
  METEOR    … 0.3131
  BERTScore … 

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
# ── Download trained model checkpoints (optional) ─────────────────────────
# Zips models/t5_outline and models/bart_story, then triggers browser download
import pathlib
from google.colab import files

MODELS_TO_DOWNLOAD = [
    ("models/t5_outline",   "t5_outline_roc.zip"),
    ("models/bart_story",   "bart_story_roc.zip"),
    # ("models/t5_outline_wp",  "t5_outline_wp.zip"),   # uncomment for WP
    # ("models/bart_story_wp",  "bart_story_wp.zip"),
]

for src, zip_name in MODELS_TO_DOWNLOAD:
    if pathlib.Path(src).exists():
        !zip -r {zip_name} {src} -q
        files.download(zip_name)
        print(f"Downloading {zip_name}")
    else:
        print(f"Skipping {src} — not found (not trained yet?)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>